In [ ]:
"""Run Phase 1 exploratory data analysis for the raw support-ticket CSV."""

from __future__ import annotations


import argparse
import hashlib
import re
from collections import Counter
from difflib import SequenceMatcher
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from langdetect import DetectorFactory, LangDetectException, detect

DetectorFactory.seed = 0
TEXT_CANDIDATES = ("issue_description", "ticket_text", "description", "text", "body", "message")
LABEL_CANDIDATES = ("category", "label", "class", "priority", "sentiment")


def find_column(columns: pd.Index, candidates: tuple[str, ...]) -> str | None:
    """Return the first case-insensitive matching column name."""
    lookup = {str(column).lower(): str(column) for column in columns}
    return next((lookup[name] for name in candidates if name in lookup), None)


def normalize_text(value: object) -> str:
    """Normalize text for deterministic duplicate detection."""
    return re.sub(r"\s+", " ", str(value).lower()).strip()


def print_overview(df: pd.DataFrame) -> pd.Series:
    """Print and return the required structural and missing-data information."""
    missing = df.isna().sum()
    print(f"Shape: {df.shape}")
    print("\nData types:\n", df.dtypes.to_string())
    print("\nMissing values per column:\n", missing.to_string())
    return missing


def plot_category_counts(df: pd.DataFrame, output_dir: Path) -> tuple[str | None, int]:
    """Plot counts for the first available existing category or label column."""
    column = find_column(df.columns, LABEL_CANDIDATES)
    if column is None:
        print("No category/label column found; skipping category chart.")
        return None, 0
    counts = df[column].fillna("<missing>").astype(str).value_counts()
    ax = counts.plot.bar(figsize=(10, 5), color="#4C78A8")
    ax.set(title=f"Ticket count by {column}", xlabel=column, ylabel="Tickets")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(output_dir / "eda_category_counts.png", dpi=150)
    plt.close()
    return column, len(counts)


def plot_text_lengths(df: pd.DataFrame, text_column: str, output_dir: Path) -> pd.Series:
    """Plot the distribution of ticket-text length measured in words."""
    lengths = df[text_column].fillna("").astype(str).str.findall(r"\b\w+\b").str.len()
    plt.figure(figsize=(10, 5))
    plt.hist(lengths, bins=50, color="#72B7B2", edgecolor="white")
    plt.title("Ticket text length (words)")
    plt.xlabel("Words")
    plt.ylabel("Tickets")
    plt.tight_layout()
    plt.savefig(output_dir / "eda_text_length_histogram.png", dpi=150)
    plt.close()
    return lengths


def duplicate_reports(df: pd.DataFrame, text_column: str, output_dir: Path) -> tuple[int, int, int]:
    """Save exact, normalized-text, and fuzzy near-duplicate candidate reports."""
    exact_mask = df.duplicated(keep=False)
    df.loc[exact_mask].to_csv(output_dir / "eda_exact_duplicate_rows.csv", index=False)
    normalized = df[text_column].fillna("").map(normalize_text)
    hashes = normalized.map(lambda text: hashlib.sha256(text.encode()).hexdigest())
    hash_mask = hashes.duplicated(keep=False) & normalized.ne("")
    df.loc[hash_mask].assign(text_hash=hashes[hash_mask]).to_csv(
        output_dir / "eda_text_hash_duplicate_rows.csv", index=False
    )
    candidates: list[dict[str, object]] = []
    blocks = pd.DataFrame({"text": normalized, "length": normalized.str.len()})
    blocks["block"] = blocks["text"].str[:16] + ":" + (blocks["length"] // 20).astype(str)
    for _, group in blocks[blocks["text"].ne("")].groupby("block"):
        ordered = group.sort_values("text")
        for (left_index, left), (right_index, right) in zip(ordered.iloc[:-1].iterrows(), ordered.iloc[1:].iterrows()):
            score = SequenceMatcher(None, left.text, right.text).ratio()
            if score >= 0.93 and left.text != right.text:
                candidates.append({"row_a": left_index, "row_b": right_index, "similarity": round(score, 3)})
    pd.DataFrame(candidates).to_csv(output_dir / "eda_near_duplicate_candidates.csv", index=False)
    return int(exact_mask.sum()), int(hash_mask.sum()), len(candidates)


def language_report(df: pd.DataFrame, text_column: str, sample_size: int) -> tuple[int, int, float]:
    """Use langdetect to estimate or measure likely non-English ticket text."""
    text = df[text_column].dropna().astype(str)
    text = text[text.str.strip().str.len().gt(0)]
    if sample_size and len(text) > sample_size:
        text = text.sample(sample_size, random_state=42)
    languages = []
    for value in text:
        try:
            languages.append(detect(value))
        except LangDetectException:
            languages.append("unknown")
    counts = Counter(languages)
    non_english = sum(count for language, count in counts.items() if language != "en")
    percent = (non_english / len(languages) * 100) if languages else 0.0
    print(f"Likely non-English: {non_english}/{len(languages)} ({percent:.2f}%)")
    return non_english, len(languages), percent


def write_summary(output_dir: Path, df: pd.DataFrame, missing: pd.Series, label: str | None,
                  label_count: int, lengths: pd.Series, duplicates: tuple[int, int, int],
                  language: tuple[int, int, float], sampled: bool) -> None:
    """Write a concise Markdown record of the EDA findings."""
    exact, text_hash, near = duplicates
    non_english, checked, percentage = language
    lines = ["# EDA Summary", "", f"- Dataset shape: **{df.shape[0]:,} rows × {df.shape[1]} columns**.",
             f"- Text-length median: **{lengths.median():.0f} words**; mean: **{lengths.mean():.1f} words**.",
             f"- Exact duplicate rows: **{exact:,}**; normalized-text hash duplicates: **{text_hash:,}**; near-duplicate candidates: **{near:,}**.",
             f"- Likely non-English texts: **{non_english:,}/{checked:,} ({percentage:.2f}%)**" + (" (sample estimate)." if sampled else ".")]
    lines.append(f"- Label chart: **{label}** with **{label_count}** values." if label else "- No category/label column was found.")
    lines.extend(["", "## Missing values", "", "| Column | Missing |", "|---|---:|"])
    lines.extend(f"| {column} | {count:,} |" for column, count in missing.items())
    (output_dir / "eda_summary.md").write_text("\n".join(lines) + "\n", encoding="utf-8")


def main() -> None:
    parser = argparse.ArgumentParser(description="Run EDA for Clario support-ticket data.")
    
    parser.add_argument(
    "--input",
    type=Path,
    default=Path("/kaggle/input/datasets/mirzayasirabdullah07/customer-support-tickets-dataset-200k-records/customer_support_tickets_200k.csv"),
    )
    
    parser.add_argument(
    "--output-dir",
    type=Path,
    default=Path("/kaggle/working/"),
    )
    
    parser.add_argument("--language-sample-size", type=int, default=0, help="0 checks every non-empty row.")
    args = parser.parse_args([])
    args.output_dir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(args.input)
    text_column = find_column(df.columns, TEXT_CANDIDATES)
    if text_column is None:
        raise ValueError(f"No ticket-text column found; checked: {', '.join(TEXT_CANDIDATES)}")
    missing = print_overview(df)
    label, label_count = plot_category_counts(df, args.output_dir)
    lengths = plot_text_lengths(df, text_column, args.output_dir)
    duplicates = duplicate_reports(df, text_column, args.output_dir)
    language = language_report(df, text_column, args.language_sample_size)
    write_summary(args.output_dir, df, missing, label, label_count, lengths, duplicates, language,
                  bool(args.language_sample_size and len(df[text_column].dropna()) > args.language_sample_size))


if __name__ == "__main__":
    main()


ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
"""
EDA: Relationship between `issue_complexity_score` and `priority`
in curated_lms_tickets.csv

Run:
    python complexity_priority_eda.py --csv /path/to/curated_lms_tickets.csv --outdir ./eda_output
"""

from __future__ import annotations

import argparse
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

PRIORITY_ORDER = ["Low", "Medium", "High", "Urgent"]


def load_data(csv_path: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["priority"] = pd.Categorical(df["priority"], categories=PRIORITY_ORDER, ordered=True)
    return df


def summary_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Mean / median / std / count of complexity score per priority level."""
    return (
        df.groupby("priority", observed=True)["issue_complexity_score"]
        .agg(["mean", "median", "std", "count"])
        .round(2)
    )


def correlation_tests(df: pd.DataFrame) -> dict:
    """
    Quantify association between the ORDINAL priority levels and the
    numeric complexity score.
    """
    codes = df["priority"].cat.codes  # Low=0, Medium=1, High=2, Urgent=3
    score = df["issue_complexity_score"]

    pearson_r, pearson_p = stats.pearsonr(codes, score)
    spearman_r, spearman_p = stats.spearmanr(codes, score)

    groups = [g["issue_complexity_score"].values for _, g in df.groupby("priority", observed=True)]
    kw_h, kw_p = stats.kruskal(*groups)

    return {
        "pearson_r": pearson_r,
        "pearson_p": pearson_p,
        "spearman_rho": spearman_r,
        "spearman_p": spearman_p,
        "kruskal_H": kw_h,
        "kruskal_p": kw_p,
    }


def chi_square_on_bins(df: pd.DataFrame) -> tuple[pd.DataFrame, float, float]:
    """Bin complexity score into Low/Mid/High and test independence from priority."""
    df = df.copy()
    df["complexity_bin"] = pd.cut(
        df["issue_complexity_score"],
        bins=[0, 3, 6, 10],
        labels=["Low (1-3)", "Mid (4-6)", "High (7-10)"],
    )
    ct = pd.crosstab(df["complexity_bin"], df["priority"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    return ct, chi2, p


def make_plots(df: pd.DataFrame, outdir: Path) -> None:
    outdir.mkdir(parents=True, exist_ok=True)
    sns.set_style("whitegrid")

    # 1) Box + strip plot of complexity score by priority
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.boxplot(data=df, x="priority", y="issue_complexity_score", order=PRIORITY_ORDER, ax=ax, palette="viridis")
    ax.set_title("Complexity Score Distribution by Priority")
    ax.set_xlabel("Priority")
    ax.set_ylabel("Issue Complexity Score")
    fig.tight_layout()
    fig.savefig(outdir / "boxplot_complexity_by_priority.png", dpi=150)
    plt.close(fig)

    # 2) Mean complexity score with error bars, by priority
    means = df.groupby("priority", observed=True)["issue_complexity_score"].mean()
    stds = df.groupby("priority", observed=True)["issue_complexity_score"].std()
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(PRIORITY_ORDER, means.reindex(PRIORITY_ORDER), yerr=stds.reindex(PRIORITY_ORDER), capsize=5, color="#4C72B0")
    ax.set_title("Mean Complexity Score by Priority (± 1 SD)")
    ax.set_ylabel("Mean Complexity Score")
    fig.tight_layout()
    fig.savefig(outdir / "meanbar_complexity_by_priority.png", dpi=150)
    plt.close(fig)

    # 3) Heatmap of complexity-bin vs priority (normalized within complexity bin)
    ct, _, _ = chi_square_on_bins(df)
    ct_norm = ct.div(ct.sum(axis=1), axis=0)
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(ct_norm, annot=True, fmt=".0%", cmap="Blues", ax=ax)
    ax.set_title("Priority Mix within Each Complexity Band (row %)")
    fig.tight_layout()
    fig.savefig(outdir / "heatmap_complexity_bin_vs_priority.png", dpi=150)
    plt.close(fig)

    # 4) Overlaid histograms / KDE
    fig, ax = plt.subplots(figsize=(8, 5))
    for p in PRIORITY_ORDER:
        sns.kdeplot(df.loc[df["priority"] == p, "issue_complexity_score"], label=p, ax=ax, fill=True, alpha=0.25)
    ax.set_title("Complexity Score Density by Priority")
    ax.set_xlabel("Issue Complexity Score")
    ax.legend(title="Priority")
    fig.tight_layout()
    fig.savefig(outdir / "kde_complexity_by_priority.png", dpi=150)
    plt.close(fig)


CSV_PATH = "/kaggle/input/datasets/ranugaweerasekara/lms-dataset/curated_lms_tickets.csv"
OUTDIR = Path("/kaggle/working/eda_output")

df = load_data(CSV_PATH)

print("=== Summary stats: complexity score per priority ===")
print(summary_stats(df))

print("\n=== Correlation / association tests ===")
for k, v in correlation_tests(df).items():
    print(f"{k}: {v:.4g}")

print("\n=== Chi-square test (binned complexity vs priority) ===")
ct, chi2, p = chi_square_on_bins(df)
print(ct)
print(f"chi2 = {chi2:.2f}, p = {p:.4g}")

make_plots(df, OUTDIR)
print(f"\nPlots saved to: {OUTDIR.resolve()}")
